### Cosine Similarity With OpenAI Embeddings

In [1]:
import numpy as np

In [2]:
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings(model="text-embedding-3-small")
embeddings

c:\Users\91767\Desktop\Practice\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000001E238F93B50>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000001E25601CC50>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [3]:
# Example 1: Finding similar sentences
sentences = [
    "The cat sat on the mat",
    "A feline rested on the rug",
    "The dog played in the yard",
    "I love programming in Python",
    "Python is my favorite programming language"
]

In [11]:
def cosine_similarity(vec1, vec2):
    dot_product=np.dot(vec1,vec2)
    norm_a=np.linalg.norm(vec1)
    norm_b=np.linalg.norm(vec2)
    return dot_product/(norm_a * norm_b)

In [5]:
sentence_embeddings=embeddings.embed_documents(sentences)
sentence_embeddings

[[-0.030763709917664528,
  -0.04955800995230675,
  -0.005008797161281109,
  -0.0015373775968328118,
  0.03621844947338104,
  -0.0020261392928659916,
  -0.008925353176891804,
  0.027170300483703613,
  0.007064019795507193,
  -0.01185307651758194,
  0.04164734110236168,
  -0.0014040789101272821,
  0.045163191854953766,
  0.05273778736591339,
  0.032004598528146744,
  0.03246993198990822,
  -0.012389502488076687,
  0.0030973756220191717,
  -0.06602564454078674,
  0.04746400937438011,
  0.025838930159807205,
  -0.045370008796453476,
  -0.003477074671536684,
  0.01459337305277586,
  0.009099853225052357,
  0.014813113957643509,
  -0.011180927976965904,
  -0.012059890665113926,
  0.010760835371911526,
  0.012854835949838161,
  0.012266705743968487,
  -0.036063339561223984,
  -0.026498153805732727,
  -0.0453958585858345,
  -0.03458978235721588,
  0.004782593343406916,
  -0.01989300362765789,
  -0.011743205599486828,
  -0.04200926795601845,
  -0.022878892719745636,
  -0.03632185980677605,
  -0

In [7]:
## Calculating the simialrity between all pairs

for i in range(len(sentences)):
    for j in range(i+1,len(sentences)):
        similarity=cosine_similarity(sentence_embeddings[i],sentence_embeddings[j])

        print(f"'{sentences[i]}' vs '{sentences[j]}'")
        print(f"Similarity: {similarity:.3f}\n")


'The cat sat on the mat' vs 'A feline rested on the rug'
Similarity: 0.656

'The cat sat on the mat' vs 'The dog played in the yard'
Similarity: 0.324

'The cat sat on the mat' vs 'I love programming in Python'
Similarity: 0.090

'The cat sat on the mat' vs 'Python is my favorite programming language'
Similarity: 0.120

'A feline rested on the rug' vs 'The dog played in the yard'
Similarity: 0.296

'A feline rested on the rug' vs 'I love programming in Python'
Similarity: 0.055

'A feline rested on the rug' vs 'Python is my favorite programming language'
Similarity: 0.103

'The dog played in the yard' vs 'I love programming in Python'
Similarity: 0.126

'The dog played in the yard' vs 'Python is my favorite programming language'
Similarity: 0.085

'I love programming in Python' vs 'Python is my favorite programming language'
Similarity: 0.708



In [8]:
## Semantic Search- Retireve the similar sentence
documents = [
    "LangChain is a framework for developing applications powered by language models",
    "Python is a high-level programming language",
    "Machine learning is a subset of artificial intelligence",
    "Embeddings convert text into numerical vectors",
    "The weather today is sunny and warm"
]
query="What is Langchain?"

In [9]:
def semantic_search(query,documents,embeddings_models,top_k=3):

    ## embed query and doument

    query_embedding=embeddings_models.embed_query(query)
    doc_embeddings = embeddings_models.embed_documents(documents)

    ## Calculate the similarity score

    similarties=[]

    for i,doc_emb in enumerate(doc_embeddings):
        similarity=cosine_similarity(query_embedding,doc_emb)
        similarties.append((similarity,documents[i]))

    ## Sort by similarity
    similarties.sort(reverse=True)
    return similarties[:top_k]

In [10]:
results=semantic_search(query,documents,embeddings)
results

[(np.float64(0.6756029006033841),
  'LangChain is a framework for developing applications powered by language models'),
 (np.float64(0.13031421770165688),
  'Python is a high-level programming language'),
 (np.float64(0.1010753259927004),
  'Embeddings convert text into numerical vectors')]

In [12]:
print(f"\nSemantic Search Results for: '{query}'")
for score, doc in results:
    print(f"Score: {score:.3f} | {doc}")


Semantic Search Results for: 'What is Langchain?'
Score: 0.676 | LangChain is a framework for developing applications powered by language models
Score: 0.130 | Python is a high-level programming language
Score: 0.101 | Embeddings convert text into numerical vectors


In [13]:
query="What is Embeddings?"
results=semantic_search(query,documents,embeddings)
results

[(np.float64(0.6227668553466031),
  'Embeddings convert text into numerical vectors'),
 (np.float64(0.2519936932917088),
  'Machine learning is a subset of artificial intelligence'),
 (np.float64(0.229052563679672),
  'LangChain is a framework for developing applications powered by language models')]